# Convert NetCDF To Zarr - Example 2

## This notebook shows a more detailed example of how the so-fresh data was processed

In [ ]:
# 1. Define your connection info
host = "eodata-bec.icm.csic.es"
port = 25288 
username = ""
password = ""

In [9]:
import paramiko

In [10]:
# 2. Initialize the SSH client
ssh = paramiko.SSHClient()

# Automatically add the server's host key (prevents "unknown host" errors)
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

# Connect to the server
ssh.connect(hostname=host, port=port, username=username, password=password)

# Open the SFTP session on the SSH connection
sftp = ssh.open_sftp()
print(f"Successfully connected to {host} via SFTP!\n")

# 3. List the files in the current directory
print("Files currently on the server:")
server_files = sftp.listdir() # listdir() returns a list of file names
for file_name in server_files:
    print(f" - {file_name}")
print("\n")

Successfully connected to eodata-bec.icm.csic.es via SFTP!

Files currently on the server:
 - LAND
 - OCEAN




In [11]:
root_path = 'OCEAN/SSS/SMOS/SouthernOcean/v1.0/L3/9day/'

In [12]:
import os

for folder in sorted(sftp.listdir(root_path)):
    if folder.isalnum() and folder != '2011':
        os.mkdir(f'./data/{folder}/')
        print('Processing year :' + folder)
        print('Downloading : ', len(sorted(sftp.listdir(root_path + folder))))
        for f in sorted(sftp.listdir(root_path + folder)):
            sftp.get(root_path + folder + '/' + f, f'./data/{folder}/{f}')

Processing year :2012
Processing year :2013
Processing year :2014
Processing year :2015
Processing year :2016
Processing year :2017
Processing year :2018
Processing year :2019
Processing year :2020
Processing year :2021
Processing year :2022
Processing year :2023


In [5]:
import xarray as xr
xr.open_dataset('./data/2011/BEC_SSS___SMOS__SO__L3__B_20110208T120000_25km__9d_REP_v100.nc')

<xarray.Dataset> Size: 8MB
Dimensions:          (time: 1, y: 720, x: 720)
Coordinates:
  * time             (time) datetime64[ns] 8B 2011-02-04
  * y                (y) float32 3kB 8.988e+06 8.962e+06 ... -8.988e+06
  * x                (x) float32 3kB -8.988e+06 -8.962e+06 ... 8.988e+06
    lat              (y, x) float32 2MB ...
    lon              (y, x) float32 2MB ...
Data variables:
    crs              int32 4B ...
    sss              (time, y, x) float32 2MB ...
    uncertainty_sss  (time, y, x) float32 2MB ...
Attributes: (12/35)
    date_created:             2023-07-12 06:19:30 GMT
    geospatial_x_units:       m
    geospatial_y_units:       m
    geospatial_x_resolution:  25000.0
    geospatial_y_resolution:  25000.0
    geospatial_x_min:         -9e+06
    ...                       ...
    copyright:                If this data is used for publication, the follo...
    comment:                  This data was produced at BEC as part of the So...
    time_coverage_start:      20110204T00:00:00
    time_coverage_end:        20110212T23:59:59
    DOI:                      10.20350/digitalCSIC/15493
    title:                    Southern Ocean SMOS Level 3 Sea Surface Salinit...

### Merge data into zarr

In [40]:
import re
import numpy as np
import xarray as xr
from typing import Iterable, List, Sequence
from datetime import datetime, timedelta
from pathlib import Path

FILENAME_RE = re.compile(r"_(\d{8})T120000_")


input_root = Path('./data/')

In [42]:
files = sorted(input_root.glob("[0-9][0-9][0-9][0-9]/*.nc"))
len(files)

4441

In [43]:
def file_center_datetime(file_path: Path) -> datetime:
    match = FILENAME_RE.search(file_path.name)
    if not match:
        raise ValueError(f"Cannot parse date from filename: {file_path.name}")
    return datetime.strptime(match.group(1), "%Y%m%d")


def build_time_bounds(files: Iterable[Path]) -> np.ndarray:
    starts: List[np.datetime64] = []
    ends: List[np.datetime64] = []
    for f in files:
        center = file_center_datetime(f)
        start = center
        end = center + timedelta(days=8, hours=23, minutes=59, seconds=59)
        starts.append(np.datetime64(start, "ns"))
        ends.append(np.datetime64(end, "ns"))
    return np.stack([starts, ends], axis=1)


def open_dataset(
    files: Sequence[Path],
    t_chunk: int = 10,
    y_chunk: int = 720,
    x_chunk: int = 720,
    parallel: bool = True,
    engine: str = 'netcdf4',
) -> xr.Dataset:
    ds = xr.open_mfdataset(
        [str(f) for f in files],
        combine="nested",
        concat_dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=parallel,
        engine=engine,
        chunks={"time": 1, "y": 720, "x": 720},
    ).sortby("time")

    ds = ds.chunk({"time": t_chunk, "y": y_chunk, "x": x_chunk})

    time_bnds = build_time_bounds(files)
    ds["time_bnds"] = xr.DataArray(
        time_bnds,
        dims=("time", "nv"),
        attrs={
            "long_name": "time bounds",
        },
    )
    ds["time"].attrs["bounds"] = "time_bnds"
    return ds

In [ ]:
ds = open_dataset(files=files)
ds.attrs['time_coverage_end'] = ds.time.values[-1]
ds

<xarray.Dataset> Size: 4GB
Dimensions:          (time: 1000, y: 720, x: 720, nv: 2)
Coordinates:
  * time             (time) datetime64[ns] 8kB 2011-02-01 ... 2013-10-28
  * y                (y) float32 3kB 8.988e+06 8.962e+06 ... -8.988e+06
  * x                (x) float32 3kB -8.988e+06 -8.962e+06 ... 8.988e+06
    lat              (y, x) float32 2MB dask.array<chunksize=(720, 720), meta=np.ndarray>
    lon              (y, x) float32 2MB dask.array<chunksize=(720, 720), meta=np.ndarray>
Dimensions without coordinates: nv
Data variables:
    crs              int32 4B ...
    sss              (time, y, x) float32 2GB dask.array<chunksize=(10, 720, 720), meta=np.ndarray>
    uncertainty_sss  (time, y, x) float32 2GB dask.array<chunksize=(10, 720, 720), meta=np.ndarray>
    time_bnds        (time, nv) datetime64[ns] 16kB 2011-02-05 ... 2013-11-09...
Attributes: (12/35)
    date_created:             2023-07-12 06:14:00 GMT
    geospatial_x_units:       m
    geospatial_y_units:       m
    geospatial_x_resolution:  25000.0
    geospatial_y_resolution:  25000.0
    geospatial_x_min:         -9e+06
    ...                       ...
    copyright:                If this data is used for publication, the follo...
    comment:                  This data was produced at BEC as part of the So...
    time_coverage_start:      20110201T00:00:00
    time_coverage_end:        2013-10-28T00:00:00.000000000
    DOI:                      10.20350/digitalCSIC/15493
    title:                    Southern Ocean SMOS Level 3 Sea Surface Salinit...

In [13]:
outputdir = 'sofresh.zarr'
ds.to_zarr(
    outputdir,
    mode='w',
    zarr_format = 3,
    consolidated=True

)

/home/krasen/data/sofresh/.pixi/envs/default/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


### Test values are all the same

In [ ]:
ds = xr.open_zarr(outputdir)
ds

In [38]:
rnd_file = np.random.choice(files[:1000], size=1)[0]
orig = xr.open_dataset(rnd_file)
orig

<xarray.Dataset> Size: 8MB
Dimensions:          (time: 1, y: 720, x: 720)
Coordinates:
  * time             (time) datetime64[ns] 8B 2012-07-11T23:00:00
  * y                (y) float32 3kB 8.988e+06 8.962e+06 ... -8.988e+06
  * x                (x) float32 3kB -8.988e+06 -8.962e+06 ... 8.988e+06
    lat              (y, x) float32 2MB ...
    lon              (y, x) float32 2MB ...
Data variables:
    crs              int32 4B ...
    sss              (time, y, x) float32 2MB ...
    uncertainty_sss  (time, y, x) float32 2MB ...
Attributes: (12/35)
    date_created:             2023-07-13 00:20:11 GMT
    geospatial_x_units:       m
    geospatial_y_units:       m
    geospatial_x_resolution:  25000.0
    geospatial_y_resolution:  25000.0
    geospatial_x_min:         -9e+06
    ...                       ...
    copyright:                If this data is used for publication, the follo...
    comment:                  This data was produced at BEC as part of the So...
    time_coverage_start:      20120712T00:00:00
    time_coverage_end:        20120720T23:59:59
    DOI:                      10.20350/digitalCSIC/15493
    title:                    Southern Ocean SMOS Level 3 Sea Surface Salinit...

In [39]:
assert np.isclose(ds.sel(time=orig.time.values[0]).sss.values, orig.sss.values, equal_nan=True).all()
assert np.isclose(ds.sel(time=orig.time.values[0]).uncertainty_sss.values, orig.uncertainty_sss.values, equal_nan=True).all()
